# Machine Learning-Based Stroke Prediction: Evaluating the Impact of Data Preprocessing Techniques

## 04_Preprocessing: Preparing the Dataset for Machine Learning

## Introduction

### Research Objective

The objective of this notebook is to prepare the stroke dataset for machine learning modelling using the most appropriate data cleaning strategies identified in the previous preprocessing experiments. This stage converts the raw healthcare data into machine-learning-ready train and test datasets while preserving methodological transparency.

This notebook does not train machine learning models. Instead, it focuses on selected missing-value handling, outlier treatment strategy, feature identification, one-hot encoding, optional feature scaling, and stratified train/test splitting. These steps are essential because the quality and structure of input data strongly influence the validity of later model comparisons.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "healthcare-dataset-stroke-data.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "results" / "tables"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20

df = pd.read_csv(DATA_PATH)
df["bmi"] = pd.to_numeric(df["bmi"], errors="coerce")

print(f"Raw dataset shape: {df.shape}")
display(df.head())

Raw dataset shape: (5110, 12)


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


## Selected Cleaning Strategy

Notebook 02 compared four missing-value methods for BMI: row deletion, mean imputation, median imputation, and KNN imputation. Median imputation was recommended because it preserves all observations, is robust to skewness and potential outliers, and is transparent enough for a dissertation methodology.

Notebook 03 investigated IQR-based outlier treatment for BMI and average glucose level. The recommended approach was not to remove outliers automatically in the primary modelling pipeline. This is because extreme BMI or glucose values may be clinically meaningful rather than erroneous. Therefore, this notebook retains outliers in the main processed dataset and records that IQR removal should be treated as a later sensitivity analysis.

The selected primary cleaning strategy is therefore:

- Apply median imputation to missing BMI values.
- Retain BMI and glucose outliers in the main modelling dataset.
- Drop the `id` column because it is an identifier, not a predictive clinical feature.

In [2]:
clean_df = df.copy()
bmi_median = clean_df["bmi"].median()
missing_bmi_before = clean_df["bmi"].isna().sum()
clean_df["bmi"] = clean_df["bmi"].fillna(bmi_median)
missing_bmi_after = clean_df["bmi"].isna().sum()

if "id" in clean_df.columns:
    clean_df = clean_df.drop(columns=["id"])

cleaning_strategy_table = pd.DataFrame([
    {
        "preprocessing_issue": "Missing BMI values",
        "selected_strategy": "Median imputation",
        "rationale": "Preserves sample size and is robust to skewness and potential outliers.",
        "affected_records": missing_bmi_before
    },
    {
        "preprocessing_issue": "BMI and glucose outliers",
        "selected_strategy": "Retain in primary dataset",
        "rationale": "Extreme clinical values may be meaningful and should not be removed automatically.",
        "affected_records": "See notebook 03 sensitivity analysis"
    },
    {
        "preprocessing_issue": "Identifier column",
        "selected_strategy": "Drop id",
        "rationale": "The identifier has no predictive clinical meaning and may introduce noise.",
        "affected_records": 0
    }
])

cleaning_strategy_table.to_csv(TABLES_DIR / "preprocessing_selected_cleaning_strategy.csv", index=False)
clean_df.to_csv(PROCESSED_DIR / "stroke_cleaned_primary.csv", index=False)

print(f"BMI median used for imputation: {bmi_median:.2f}")
print(f"Missing BMI before imputation: {missing_bmi_before}")
print(f"Missing BMI after imputation: {missing_bmi_after}")
print(f"Cleaned primary dataset shape: {clean_df.shape}")
display(cleaning_strategy_table)

BMI median used for imputation: 28.10
Missing BMI before imputation: 201
Missing BMI after imputation: 0
Cleaned primary dataset shape: (5110, 11)


,preprocessing_issue,selected_strategy,rationale,affected_records
0,Missing BMI values,Median imputation,Preserves sample size and is robust to skewnes...,201
1,BMI and glucose outliers,Retain in primary dataset,Extreme clinical values may be meaningful and ...,See notebook 03 sensitivity analysis
2,Identifier column,Drop id,The identifier has no predictive clinical mean...,0


## Feature Identification

Before encoding and scaling, features must be separated by type. Numerical features can be used directly by most algorithms, although scaling may be beneficial. Categorical features must be converted into numeric representations before machine learning models can process them.

The target variable is `stroke`. It is excluded from feature transformation and used only for stratified splitting and later supervised modelling.

In [3]:
target_column = "stroke"
feature_df = clean_df.drop(columns=[target_column])
target = clean_df[target_column]

categorical_features = feature_df.select_dtypes(include=["object"]).columns.tolist()
numerical_features = feature_df.select_dtypes(include=["number"]).columns.tolist()

feature_identification_table = pd.DataFrame({
    "feature": categorical_features + numerical_features,
    "feature_type": ["categorical"] * len(categorical_features) + ["numerical"] * len(numerical_features)
})

feature_identification_table.to_csv(TABLES_DIR / "preprocessing_feature_identification.csv", index=False)

print("Categorical features:", categorical_features)
print("Numerical features:", numerical_features)
display(feature_identification_table)

Categorical features: ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
Numerical features: ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']


C:\Users\11315\AppData\Local\Temp\ipykernel_51672\2011317569.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = feature_df.select_dtypes(include=["object"]).columns.tolist()


,feature,feature_type
0,gender,categorical
1,ever_married,categorical
2,work_type,categorical
3,Residence_type,categorical
4,smoking_status,categorical
5,age,numerical
6,hypertension,numerical
7,heart_disease,numerical
8,avg_glucose_level,numerical
9,bmi,numerical


## One-Hot Encoding

One-hot encoding converts categorical variables into binary indicator columns. This prevents machine learning algorithms from interpreting category labels as ordinal numeric values. For example, categories such as work type or smoking status do not have a natural numerical order, so one-hot encoding is more appropriate than assigning arbitrary integer labels.

To avoid unnecessary duplication, one category from each categorical variable is dropped using `drop_first=True`. This reduces multicollinearity for linear models while preserving the information contained in the original categories.

In [4]:
encoded_features = pd.get_dummies(feature_df, columns=categorical_features, drop_first=True, dtype=int)
encoded_df = encoded_features.copy()
encoded_df[target_column] = target.values

encoding_summary = pd.DataFrame({
    "metric": [
        "Original feature count before encoding",
        "Generated feature count after encoding",
        "Categorical features encoded",
        "Numerical features retained"
    ],
    "value": [
        feature_df.shape[1],
        encoded_features.shape[1],
        len(categorical_features),
        len(numerical_features)
    ]
})

encoded_feature_list = pd.DataFrame({"encoded_feature": encoded_features.columns})
encoding_summary.to_csv(TABLES_DIR / "preprocessing_encoding_summary.csv", index=False)
encoded_feature_list.to_csv(TABLES_DIR / "preprocessing_encoded_feature_list.csv", index=False)
encoded_df.to_csv(PROCESSED_DIR / "stroke_encoded_full.csv", index=False)

display(encoding_summary)
display(encoded_feature_list)

,metric,value
0,Original feature count before encoding,10
1,Generated feature count after encoding,16
2,Categorical features encoded,5
3,Numerical features retained,5


,encoded_feature
0,age
1,hypertension
2,heart_disease
3,avg_glucose_level
4,bmi
5,gender_Male
6,gender_Other
7,ever_married_Yes
8,work_type_Never_worked
9,work_type_Private


## Feature Scaling

Two preprocessing variants are created for later modelling comparison:

1. **No Scaling**: encoded features are retained on their original numeric scales.
2. **StandardScaler**: numerical features are transformed to have a mean of approximately zero and a standard deviation of one.

Scaling can affect machine learning models because many algorithms are sensitive to the magnitude of input features. Distance-based methods such as K-nearest neighbours and margin-based methods such as support vector machines can be dominated by variables with larger numeric ranges. Regularised linear models can also be affected because penalty terms operate on coefficient sizes that depend on feature scale. Tree-based methods, however, are generally less sensitive to scaling because they split features by thresholds rather than distances or coefficient magnitudes.

In [5]:
X_no_scaling = encoded_features.copy()
X_standard_scaled = encoded_features.copy()

scaler = StandardScaler()
X_standard_scaled[numerical_features] = scaler.fit_transform(X_standard_scaled[numerical_features])

no_scaling_full = X_no_scaling.copy()
no_scaling_full[target_column] = target.values

standard_scaled_full = X_standard_scaled.copy()
standard_scaled_full[target_column] = target.values

no_scaling_full.to_csv(PROCESSED_DIR / "stroke_preprocessed_no_scaling_full.csv", index=False)
standard_scaled_full.to_csv(PROCESSED_DIR / "stroke_preprocessed_standard_scaled_full.csv", index=False)

scaling_comparison = pd.DataFrame([
    {
        "variant": "No Scaling",
        "description": "Encoded features retained on original numeric scales.",
        "best_suited_models": "Tree-based methods and baseline comparison.",
        "limitation": "Scale-sensitive models may be dominated by large-range variables."
    },
    {
        "variant": "StandardScaler",
        "description": "Numerical features standardised to zero mean and unit variance.",
        "best_suited_models": "Logistic regression, SVM, KNN, neural networks, and other scale-sensitive methods.",
        "limitation": "Less necessary for tree-based models and slightly less interpretable in raw units."
    }
])

scaling_comparison.to_csv(TABLES_DIR / "preprocessing_scaling_comparison.csv", index=False)
display(scaling_comparison)

,variant,description,best_suited_models,limitation
0,No Scaling,Encoded features retained on original numeric ...,Tree-based methods and baseline comparison.,Scale-sensitive models may be dominated by lar...
1,StandardScaler,Numerical features standardised to zero mean a...,"Logistic regression, SVM, KNN, neural networks...",Less necessary for tree-based models and sligh...


## Train/Test Split

The dataset is split into training and test subsets using stratified sampling. Stratification preserves the proportion of stroke and non-stroke cases in both subsets, which is essential because the target variable is highly imbalanced. Without stratification, the minority stroke class could be underrepresented in either the training or test set, leading to unreliable model development or evaluation.

The scaler has been applied before splitting in the full comparison dataset above only to create a general preprocessing variant table. For the saved train/test datasets below, scaling parameters are fitted on the training set and then applied to the test set. This avoids information from the test set influencing the training transformation.

In [6]:
X_train_no_scaling, X_test_no_scaling, y_train, y_test = train_test_split(
    X_no_scaling,
    target,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=target
)

train_scaler = StandardScaler()
X_train_standard_scaled = X_train_no_scaling.copy()
X_test_standard_scaled = X_test_no_scaling.copy()
X_train_standard_scaled[numerical_features] = train_scaler.fit_transform(X_train_standard_scaled[numerical_features])
X_test_standard_scaled[numerical_features] = train_scaler.transform(X_test_standard_scaled[numerical_features])

train_no_scaling = X_train_no_scaling.copy()
train_no_scaling[target_column] = y_train.values
test_no_scaling = X_test_no_scaling.copy()
test_no_scaling[target_column] = y_test.values

train_standard_scaled = X_train_standard_scaled.copy()
train_standard_scaled[target_column] = y_train.values
test_standard_scaled = X_test_standard_scaled.copy()
test_standard_scaled[target_column] = y_test.values

train_no_scaling.to_csv(PROCESSED_DIR / "stroke_train_no_scaling.csv", index=False)
test_no_scaling.to_csv(PROCESSED_DIR / "stroke_test_no_scaling.csv", index=False)
train_standard_scaled.to_csv(PROCESSED_DIR / "stroke_train_standard_scaled.csv", index=False)
test_standard_scaled.to_csv(PROCESSED_DIR / "stroke_test_standard_scaled.csv", index=False)

split_summary = pd.DataFrame([
    {
        "dataset": "Full encoded dataset",
        "rows": len(encoded_df),
        "stroke_cases": int(encoded_df[target_column].sum()),
        "stroke_percentage": round(encoded_df[target_column].mean() * 100, 2)
    },
    {
        "dataset": "Training set",
        "rows": len(train_no_scaling),
        "stroke_cases": int(train_no_scaling[target_column].sum()),
        "stroke_percentage": round(train_no_scaling[target_column].mean() * 100, 2)
    },
    {
        "dataset": "Test set",
        "rows": len(test_no_scaling),
        "stroke_cases": int(test_no_scaling[target_column].sum()),
        "stroke_percentage": round(test_no_scaling[target_column].mean() * 100, 2)
    }
])

final_dataset_summary = pd.DataFrame({
    "metric": [
        "Final dataset rows",
        "Final dataset columns including target",
        "Generated features after encoding",
        "Training rows",
        "Test rows"
    ],
    "value": [
        encoded_df.shape[0],
        encoded_df.shape[1],
        encoded_features.shape[1],
        train_no_scaling.shape[0],
        test_no_scaling.shape[0]
    ]
})

split_summary.to_csv(TABLES_DIR / "preprocessing_train_test_split_summary.csv", index=False)
final_dataset_summary.to_csv(TABLES_DIR / "preprocessing_final_dataset_summary.csv", index=False)

display(split_summary)
display(final_dataset_summary)

,dataset,rows,stroke_cases,stroke_percentage
0,Full encoded dataset,5110,249,4.87
1,Training set,4088,199,4.87
2,Test set,1022,50,4.89


,metric,value
0,Final dataset rows,5110
1,Final dataset columns including target,17
2,Generated features after encoding,16
3,Training rows,4088
4,Test rows,1022


## Discussion

The preprocessing workflow applies the cleaning decisions from the previous notebooks while preserving separate variants for later modelling experiments. Median imputation provides a robust and interpretable solution for missing BMI values. Outliers are retained in the primary dataset because the previous outlier analysis showed that IQR removal can remove a substantial number of records, and extreme BMI or glucose values may be clinically meaningful.

One-hot encoding allows categorical variables to be used by machine learning algorithms without imposing artificial ordinal relationships. The comparison between no scaling and StandardScaler prepares the project for fair model experimentation. Scale-sensitive models should use the standard-scaled datasets, while tree-based models can be evaluated using either variant.

## Conclusion

This notebook prepared the stroke dataset for machine learning by applying median BMI imputation, retaining clinically plausible outliers in the primary dataset, identifying feature types, applying one-hot encoding, creating no-scaling and StandardScaler dataset variants, and splitting the data into stratified training and test sets.

The saved outputs provide a reproducible foundation for the modelling stage. Later notebooks should train and compare models using these prepared datasets while maintaining awareness of class imbalance and the methodological effects of scaling.